<center> 
    
## **Predicting Heart Attack Risk Using Patient Health Indicators** 

</center>

## Project Introduction

Heart disease is a global health concern and one of the leading causes of death worldwide. Among various types of cardiovascular diseases, heart attacks represent a critical medical emergency that can often be prevented through early detection and lifestyle modification. With the growing availability of healthcare data, predictive analytics and machine learning have become powerful tools in identifying individuals at high risk before an adverse event occurs.

## Dataset Selection

The objective of this project is to design a predictive model capable of estimating a person’s **heart attack risk** based on their demographic, medical, and lifestyle characteristics. I have decided to choose dataset from `https://www.kaggle.com/datasets/iamsouravbanerjee/heart-attack-prediction-dataset` since it contains data for 8,700+ records, which is large enough to preform Exploratory Data Analysis (EDA). The dataset used includes 26 variables such as **age, sex, cholesterol, blood pressure, heart rate, stress level, BMI, smoking habits, exercise frequency, and previous heart conditions**, as well as the target variable **“Heart Attack Risk”** (0 = low risk, 1 = high risk).

### Business Problem

The business problem being addressed is:
1. How does heart attack risk vary across different age groups? Are older patients more likely to be at risk?
2. Does high cholesterol correlate with a greater likelihood of heart attack risk?
3. Which health metrics are most strongly correlated with heart attack risk?
4. How do exercise habits and smoking behavior differ between high- and low-risk patients?
5. How do different machine learning techniques, including class balancing, SMOTE, and hyperparameter tuning, affect the accuracy and effectiveness of heart attack risk prediction?
### Model Target and Objective

The **target variable** in this project is:
- **Heart Attack Risk** — a binary variable representing whether a person is likely (`1`) or unlikely (`0`) to experience a heart attack based on their health and lifestyle data.

The model’s objective is to predict this target variable using input features such as cholesterol levels, stress, age, and other related factors. By identifying which variables have the strongest influence on heart attack risk, the model can also provide valuable interpretability and actionable insights.



### Project folders
The dataset is stored in `data/`, the notebook in `notebooks/`, and saved graphs and tables in `figures/` and `results/`.

In [ ]:
# Folder setup for GitHub project
from pathlib import Path
current_folder = Path.cwd().resolve()
project_folder = next((p for p in [current_folder, current_folder.parent]
                       if (p / 'data' / 'heart_attack_prediction_dataset.csv').exists()), None)
if project_folder is None:
    raise FileNotFoundError('Place the dataset in Heart-Attack-Risk-Prediction/data/')
data_folder = project_folder / 'data'
figures_folder = project_folder / 'figures'
results_folder = project_folder / 'results'
figures_folder.mkdir(exist_ok=True)
results_folder.mkdir(exist_ok=True)


In [ ]:
# Importing important libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, HTML
import warnings

# Suppressing all warnings
warnings.filterwarnings('ignore')  

# Function for organized and readable section separation
def display_section_title(title, level=1, color="#000000"):
    # Limit heading level to between h1 and h6
    level = min(max(1, level), 6)
    
    html = f"""
    <div style='text-align:left; margin:20px 0;'>
        <h{level} style='color:{color}; margin:0;'>{title}</h{level}>
    """
    html += "</div>"
    
    display(HTML(html))

display_section_title("Reading the heart_attack_prediction_dataset.csv file.", level = 5)

# Reading the heart_attack_prediction_dataset.csv file
print("Reading data from file...")
heart_df = pd.read_csv(data_folder / 'heart_attack_prediction_dataset.csv')
# Keep an unchanged copy for the modeling section
original_df = heart_df.copy()
print("Data loaded successfully. Number of rows in the dataset:", len(heart_df))
print()

display_section_title("Data cleaning and preparing.", level = 5)

# Data cleaning and preparing
print("Starting of Data cleaning and preparing...")
print()

# Handle missing values
print("Handling missing values...")

# Checking for missing values in each column of the dataset
missing_report = heart_df.isnull().sum()

# Filtering out columns that have no missing values (keep only those with > 0 missing)
missing_report = missing_report[missing_report > 0]

# If there are any missing values, display them
if not missing_report.empty:
    print("Missing values per column:\n", missing_report)
# Otherwise, indicate that there are no missing values
else:
    print("No columns has missing values.")

# Splitting Blood Pressure into systolic and diastolic columns
bp_split = heart_df["Blood Pressure"].str.split("/", expand=True)

# Converting the first part (Systolic) to a numeric column and handle any errors by coercing invalid values to NaN
heart_df["Systolic_BP"] = pd.to_numeric(bp_split[0], errors="coerce")

# Converting the second part (Diastolic) to a numeric column and handle any errors by coercing invalid values to NaN
heart_df["Diastolic_BP"] = pd.to_numeric(bp_split[1], errors="coerce")
print("Splitting of Blood Pressure into systolic and diastolic columns")

# Setting the Seaborn plotting style to 'whitegrid' for a clean background with gridlines
# and use the 'coolwarm' color palette for plots
sns.set(style="whitegrid", palette="coolwarm")

print()
print("Data cleaning and preparing is complete...")
# ============================================================
# Graph 1: Age Distribution by Heart Attack Risk
# ============================================================
display_section_title("Graph 1: Age Distribution by Heart Attack Risk", level = 5)

# Creating a new figure with specified dimensions 
plt.figure(figsize=(8,5))

# Plotting a histogram showing the distribution of 'Age', separated by 'Heart Attack Risk' levels
plt.figure(figsize=(10, 6))
ax = sns.histplot(
    data=heart_df,
    x="Age",
    hue="Heart Attack Risk",
    hue_order=[1,0],
    multiple="stack",
    bins=8,
    palette={0: "#4CAF50", 1: "#E59882"}
)

# Adding a bold title to the plot
plt.title("Age Distribution by Heart Attack Risk", fontsize=14, weight='bold')

# Labeling the x-axis
plt.xlabel("Age")

# Labeling the y-axis
plt.ylabel("Number of Patients")

# Make legend lines thicker and move legend to the right
handles, labels = ax.get_legend_handles_labels()
for h in handles:
    h.set_linewidth(4)  # make legend lines thicker

# Moving legend to the right and ensure labels are shown
plt.legend(title='Heart Attack Risk', labels=['No (0)', 'Yes (1)'], loc='center left', bbox_to_anchor=(1, 0.5))

# Displaying the plot
plt.show()

##### Graph 1 Analysis:
This histogram shows that the proportion of high-risk individuals increases significantly with age.
It is clear that **older patients (especially above age 60)** appear more frequently in the higher-risk category, confirming that age is one of the strongest predictors of heart attack risk.

In [ ]:
# ============================================================
# Graph 2: Cholesterol Levels by Heart Attack Risk
# ============================================================
display_section_title("Graph 2: Cholesterol Levels by Heart Attack Risk", level = 5)

# Creating a new figure with specified size
plt.figure(figsize=(8,5))

# Creating a box plot to compare cholesterol levels across heart attack risk categories
sns.boxplot(data=heart_df, x="Heart Attack Risk", y="Cholesterol", palette="coolwarm",hue=None)

# Add a bold title to the chart
plt.title("Cholesterol Levels by Heart Attack Risk", fontsize=14, weight='bold')

# Labeling the x-axis to clarify what 0 and 1 represent
plt.xlabel("Heart Attack Risk (0 = No, 1 = Yes)")

# Labeling the y-axis to indicate the cholesterol measurement unit
plt.ylabel("Cholesterol (mg/dL)")

# Displaying the plot
plt.show()

##### Graph 2 Analysis:
The boxplot clearly shows that patients in the high-risk category generally have higher cholesterol levels than those in the low-risk group. We see that **high-risk individuals tend to have higher cholesterol values** on average. This supports medical findings that high cholesterol contributes significantly to cardiovascular disease.

In [ ]:
# ============================================================
# Graph 3: Correlation Heatmap of Key Health Metrics
# ============================================================
display_section_title("Graph 3: Correlation Heatmap of Key Health Metrics", level = 5)

# Defining a list of numeric columns to include in the correlation analysis
numeric_cols = ["Age", "Cholesterol", "Heart Rate", "Stress Level", 
                "BMI", "Triglycerides", "Systolic_BP", "Diastolic_BP", "Heart Attack Risk"]

# Creating a new figure with specified size
plt.figure(figsize=(10,8))

# Computing the correlation matrix for the selected numeric columns
corr = heart_df[numeric_cols].corr()

# Plotting the correlation heatmap
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")

# Adding a bold title to the heatmap
plt.title("Correlation Heatmap of Key Health Metrics", fontsize=14, weight='bold')

# Displaying the plot
plt.show()

##### Graph 3 Analysis

This heatmap illustrates the relationships among the main numeric features. 
Heart Attack Risk is positively correlated with Age, Cholesterol, BMI, Stress Level, and Triglycerides.
These findings indicate that both metabolic and emotional stress factors significantly influence cardiac health.
This visualization helps identify which variables are most important to include in predictive modeling.


In [ ]:
# ============================================================
# Graph 4: Lifestyle Factors – Exercise vs Smoking by Risk
# ============================================================
display_section_title("Graph 4: Lifestyle Factors – Exercise vs Smoking by Risk", level = 5)

heart_df['Smoking_label'] = heart_df['Smoking'].map({0: 'No (0)', 1: 'Yes (1)'})

# Creating a new figure with specified size
plt.figure(figsize=(8,5))

# Reverse the coolwarm palette
palette = sns.color_palette("coolwarm", 2)[::-1]

# Creating a bar plot to compare average exercise hours per week across heart attack risk levels,
# further divided by smoking status
sns.barplot(data=heart_df, x="Heart Attack Risk", y="Exercise Hours Per Week", hue="Smoking_label", palette=palette,ci=None)

# Adding a bold title to the chart
plt.title("Exercise Hours and Smoking by Heart Attack Risk", fontsize=14, weight='bold')

# Labeling the x-axis to clarify the meaning of 0 and 1
plt.xlabel("Heart Attack Risk (0 = No, 1 = Yes)")

# Labeling the y-axis to describe the measurement
plt.ylabel("Average Exercise Hours per Week")

# Moving legend to the right and show 0/1 values
plt.legend(title="Smoking", loc='center left', bbox_to_anchor=(1, 0.5))

# Displaying the plot
plt.show()

##### Graph 4 Analysis
This bar plot compares exercise hours and smoking status by heart attack risk group. We see that **high-risk patients generally exercise less and smoke more** than low-risk individuals. This emphasizes how lifestyle choices play a crucial role in heart attack prevention.


## Overview and Conclusion of the insights gained from graphical analysis

The graphical analysis provided key insights into the factors most associated with **heart attack risk**. The data revealed that **age and cholesterol levels** have a strong relationship with increased risk — older individuals and those with higher cholesterol readings tend to appear more frequently in the high-risk category.  

The **correlation heatmap** highlighted positive relationships between heart attack risk and variables such as **stress level, triglycerides, and BMI**, indicating that metabolic and psychological factors play a significant role. In contrast, features representing **healthy habits**, such as higher exercise hours and lower smoking rates, showed inverse relationships with heart attack risk.  

These findings help us understand the most influential predictors of heart attack likelihood and form the foundation for developing predictive models in subsequent stages.

The insights from the graphical analysis suggest that both **biological indicators** (like cholesterol and age) and **lifestyle behaviors** (like smoking, stress, and exercise) contribute significantly to heart attack risk.  

Overall, the exploratory analysis confirms that **heart attack risk is multifactorial**, emphasizing the importance of a balanced lifestyle and early health monitoring. 

## Data Cleaning and Preparation

I first check for duplicate records, missing values, and data types. I also separate the blood pressure reading into systolic and diastolic values and create a BMI category from the original BMI measurements. These steps do not require learning values from the full dataset.

I will split the data into training and testing sets **before** fitting missing-value imputation, encoding, or scaling. Those steps will be handled within the model pipelines so that the test data stays independent.

In [ ]:
# Prepare the dataset for machine learning without fitting transformations
import numpy as np

df = original_df.copy()
print("Original shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(15))

# Remove exact duplicates
df = df.drop_duplicates().copy()

# Split the blood pressure column into two numeric measurements
bp = df['Blood Pressure'].astype(str).str.extract(r'^\s*(\d+)\s*/\s*(\d+)\s*$')
df['Systolic BP'] = pd.to_numeric(bp[0], errors='coerce')
df['Diastolic BP'] = pd.to_numeric(bp[1], errors='coerce')

# Make BMI categories using original values, not scaled values
df['BMI Category'] = pd.cut(df['BMI'], bins=[-np.inf,18.5,25,30,np.inf],
                            labels=['Underweight','Normal','Overweight','Obese'],
                            right=False).astype('object')

# Treat clearly invalid values as missing
df.loc[df['BMI'] <= 0, 'BMI'] = np.nan
df.loc[df['Age'] <= 0, 'Age'] = np.nan

# Patient ID is an identifier, not a health predictor
df = df.drop(columns=['Patient ID', 'Blood Pressure'])
print("Prepared shape:", df.shape)
display(df.head())

## Model Selection and Justification

I selected **Logistic Regression** as a simple, interpretable classification model and **Random Forest** to test whether nonlinear relationships between patient characteristics can improve predictions.

For each algorithm, I compare three approaches: the original imbalanced data, balanced class weights, and SMOTE. I evaluate each approach both before and after hyperparameter tuning, giving **12 model versions** in total. I also include a majority-class baseline for comparison.

## Model Building and Evaluation

I use an 80/20 stratified train/test split and five-fold stratified cross-validation for model selection. Imputation, categorical encoding, scaling, and SMOTE are performed inside the modeling pipeline to avoid data leakage. The final test set is kept unchanged for every model.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, imblearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, classification_report, precision_recall_curve)
from sklearn.inspection import permutation_importance
SEED=42
FAST_MODE=False
FOLDS=3 if FAST_MODE else 5
N_ITER=2 if FAST_MODE else 5
print('Python',sys.version.split()[0], 'sklearn',sklearn.__version__, 'imblearn',imblearn.__version__)


# Reuse the project folders defined at the beginning of this notebook
DATA_DIR = data_folder
FIGURES_DIR = figures_folder
RESULTS_DIR = results_folder

def save_figure(name):
    plt.savefig(FIGURES_DIR / name, dpi=200, bbox_inches='tight')
    print('Saved figure:', FIGURES_DIR / name)


### Step 2: Split the data into training and testing sets
I use a stratified split to keep the target proportions similar in both groups.

In [ ]:
X=df.drop(columns='Heart Attack Risk'); y=df['Heart Attack Risk'].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=SEED)
cat_cols=X_train.select_dtypes(include=['object','category']).columns.tolist()
num_cols=[c for c in X_train.columns if c not in cat_cols]
print('Training:',X_train.shape,'Test:',X_test.shape,'positive rates:',round(y_train.mean(),3),round(y_test.mean(),3))
print('Categorical:',cat_cols);print('Numeric:',num_cols)

### Step 3: Prepare the data and handle imbalance
Missing values and categorical features are handled inside the model pipeline. SMOTENC is applied only to training folds because the data contains categorical features.

In [ ]:
def make_ohe():
    try: return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore',sparse=False)

def make_pipeline(algorithm,method):
    # Stage 1: impute numeric data and ordinal-encode categorical data.
    first=ColumnTransformer([
        ('numeric',SimpleImputer(strategy='median'),num_cols),
        ('categorical',SkPipeline([('impute',SimpleImputer(strategy='most_frequent')),
                                   ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))]),cat_cols)
    ],sparse_threshold=0)
    # Stage 2: one-hot encode categorical indices after optional SMOTENC.
    n_num=len(num_cols);n_cat=len(cat_cols)
    second=ColumnTransformer([
        ('numeric',StandardScaler() if algorithm=='Logistic Regression' else 'passthrough',list(range(n_num))),
        ('categorical',make_ohe(),list(range(n_num,n_num+n_cat)))
    ],sparse_threshold=0)
    balanced=method=='Class weights'
    if algorithm=='Logistic Regression':
        clf=LogisticRegression(max_iter=2000,random_state=SEED,class_weight='balanced' if balanced else None)
    else:
        clf=RandomForestClassifier(n_estimators=100 if FAST_MODE else 150,n_jobs=-1,random_state=SEED,class_weight='balanced' if balanced else None)
    steps=[('prepare',first)]
    if method=='SMOTE':
        if n_cat:
            steps.append(('smote',SMOTENC(categorical_features=list(range(n_num,n_num+n_cat)),random_state=SEED,k_neighbors=5)))
        else:
            from imblearn.over_sampling import SMOTE
            steps.append(('smote',SMOTE(random_state=SEED,k_neighbors=5)))
    steps.extend([('features',second),('model',clf)])
    return ImbPipeline(steps)

def search_space(algorithm):
    if algorithm=='Logistic Regression':return {'model__C':[0.01,0.1,1,10,100], 'model__solver':['lbfgs']}
    return {'model__n_estimators':[80,120,180,250], 'model__max_depth':[None,5,10,20], 'model__min_samples_leaf':[1,2,4], 'model__min_samples_split':[2,5,10]}
cv=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)

### Step 4: Majority-class baseline
This simple model helps me check whether the machine learning models perform better than predicting the most common outcome.

In [ ]:
dummy=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
print(classification_report(y_test,dummy.predict(X_test),zero_division=0))

### Step 5: Compare baseline and tuned models
I test both algorithms using the original imbalanced data, class weights, and SMOTE. Each approach has a baseline and tuned version, for 12 comparisons. I use cross-validation PR-AUC to compare models.

In [ ]:
from sklearn.model_selection import cross_validate
results=[]; fitted={};predictions={}
def evaluate(name,model,algorithm,method,stage,cv_ap=np.nan):
    pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    row={'Model':algorithm,'Balancing':method,'Version':stage,'Name':name,
         'Accuracy':accuracy_score(y_test,pred),'Balanced Accuracy':balanced_accuracy_score(y_test,pred),
         'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),
         'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob),
         'PR-AUC':average_precision_score(y_test,prob),'CV PR-AUC':cv_ap,
         'False Negatives':int(confusion_matrix(y_test,pred,labels=[0,1])[1,0]),
         'False Positives':int(confusion_matrix(y_test,pred,labels=[0,1])[0,1])}
    results.append(row);fitted[name]=model;predictions[name]=(pred,prob)
    print(f"{name}: recall={row['Recall']:.3f}, PR-AUC={row['PR-AUC']:.3f}, false negatives={row['False Negatives']}")

for algorithm in ['Logistic Regression','Random Forest']:
    for method in ['Original (imbalanced)','Class weights','SMOTE']:
        baseline=make_pipeline(algorithm,method)
        baseline_cv=cross_validate(baseline,X_train,y_train,cv=cv,scoring='average_precision',n_jobs=1)
        baseline.fit(X_train,y_train)
        evaluate(f'{algorithm} | {method} | Baseline',baseline,algorithm,method,'Baseline',baseline_cv['test_score'].mean())
        tuned=RandomizedSearchCV(make_pipeline(algorithm,method),search_space(algorithm),n_iter=N_ITER,
                                 scoring='average_precision',cv=cv,random_state=SEED,n_jobs=1,refit=True,error_score='raise')
        tuned.fit(X_train,y_train)
        print('Best hyperparameters:',tuned.best_params_)
        evaluate(f'{algorithm} | {method} | Tuned',tuned.best_estimator_,algorithm,method,'Tuned',tuned.best_score_)
results_df=pd.DataFrame(results)
display(results_df.sort_values('CV PR-AUC',ascending=False).round(4))
assert len(results_df)==12
results_df.to_csv(RESULTS_DIR / 'model_comparison_12_configurations.csv', index=False)


### Step 6: Compare the results
I compare the models using accuracy, recall, precision, F1-score, ROC-AUC, PR-AUC, and confusion matrices. I pay particular attention to missed high-risk cases.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,5))
axes[0].barh(results_df['Name'],results_df['CV PR-AUC']);axes[0].set_title('Training CV average precision')
axes[1].barh(results_df['Name'],results_df['Recall']);axes[1].set_title('Holdout positive-class recall')
plt.tight_layout();save_figure('model_comparison.png');plt.show()
best_name=results_df.sort_values('CV PR-AUC',ascending=False).iloc[0]['Name']
print('CV-selected model:',best_name)
fig,axes=plt.subplots(1,3,figsize=(15,4))
ConfusionMatrixDisplay.from_predictions(y_test,predictions[best_name][0],ax=axes[0],colorbar=False)
RocCurveDisplay.from_predictions(y_test,predictions[best_name][1],ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test,predictions[best_name][1],ax=axes[2])
plt.tight_layout();save_figure('selected_model_diagnostics.png');plt.show()
print(classification_report(y_test,predictions[best_name][0],zero_division=0))

### Step 7: Check prediction thresholds
I look at how different thresholds change the number of patients identified as high risk. This is descriptive; the test set is not used to choose a new threshold.

In [ ]:
p=predictions[best_name][1]
threshold_rows=[]
for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7]:
    pred=(p>=t).astype(int)
    threshold_rows.append({'Threshold':t,'Recall':recall_score(y_test,pred,zero_division=0),
                           'Precision':precision_score(y_test,pred,zero_division=0),
                           'False Negatives':int(((y_test==1)&(pred==0)).sum()),
                           'False Positives':int(((y_test==0)&(pred==1)).sum())})
threshold_df=pd.DataFrame(threshold_rows)
display(threshold_df.round(3))
threshold_df.to_csv(RESULTS_DIR / 'threshold_sensitivity_descriptive.csv', index=False)

### Step 8: Feature importance
I use permutation importance to explore which features affect the selected model predictions. Importance does not establish a cause-and-effect relationship.

In [ ]:
importance=permutation_importance(fitted[best_name],X_train,y_train,n_repeats=3 if FAST_MODE else 5,
                                  scoring='average_precision',random_state=SEED,n_jobs=1)
imp=pd.Series(importance.importances_mean,index=X_train.columns).sort_values(ascending=False)
display(imp.head(12).to_frame('Mean importance'))
imp.head(12).sort_values().plot.barh(figsize=(8,5),title='Permutation importance (training data)');plt.tight_layout();save_figure('feature_importance.png');plt.show()
imp.rename('Mean importance').to_csv(RESULTS_DIR / 'feature_importance.csv')


### Step 9: Fairness checks
I compare results across available demographic groups where sample sizes are adequate. These checks are exploratory and cannot establish clinical fairness.

In [ ]:
fairness=[]
for group in ['Sex','Continent']:
    if group in X_test:
        for value,ids in X_test.groupby(group).groups.items():
            loc=X_test.index.get_indexer(ids)
            truth=y_test.iloc[loc]; pred=predictions[best_name][0][loc]
            fairness.append({'Attribute':group,'Group':value,'Count':len(loc),'Positive cases':int(truth.sum()),
                             'Recall':recall_score(truth,pred,zero_division=0) if truth.sum() else np.nan})
fairness_df=pd.DataFrame(fairness)
display(fairness_df.round(3))
fairness_df.to_csv(RESULTS_DIR / 'fairness_group_recall.csv', index=False)

### Ethical Considerations
Patient privacy, possible demographic bias, data quality, missed positive cases, and limitations of synthetic samples from SMOTE are important. The dataset is used for educational analysis only, not medical diagnosis. More detail will be provided in the white paper.

### Step 10: Summarize the findings
I use the model results below to explain whether balancing and tuning helped. Results should be interpreted after running the notebook.

In [ ]:
winner=results_df.sort_values('CV PR-AUC',ascending=False).iloc[0]
print(f"Selected by training CV PR-AUC: {winner['Name']}")
print(f"Training CV PR-AUC: {winner['CV PR-AUC']:.3f}")
print(f"Holdout accuracy: {winner['Accuracy']:.3f}; balanced accuracy: {winner['Balanced Accuracy']:.3f}")
print(f"Holdout recall: {winner['Recall']:.3f}; precision: {winner['Precision']:.3f}; F1: {winner['F1']:.3f}")
print(f"Holdout ROC-AUC: {winner['ROC-AUC']:.3f}; PR-AUC: {winner['PR-AUC']:.3f}")
print(f"Holdout false negatives: {winner['False Negatives']}; false positives: {winner['False Positives']}")
print(f"Holdout positive prevalence: {y_test.mean():.3f} (a useful PR-AUC reference)")
print('LIMITATIONS: Dataset origin/label validity not independently verified; possible weak signal; single random holdout; no external clinical validation.')
print('RECOMMENDATIONS: Verify dataset provenance, compare repeated splits, investigate calibration and label quality, and obtain expert review before any use beyond education.')

### Final Conclusion and Limitations
After running the notebook, I will summarize which model performed best and whether the results are strong enough for meaningful prediction. Accuracy alone is not enough when the goal is to identify high-risk cases. This is an educational project and requires external validation before any clinical use.

**Dataset source:** [Heart Attack Prediction Dataset on Kaggle](https://www.kaggle.com/datasets/iamsouravbanerjee/heart-attack-prediction-dataset).